# 데이터 전처리 기초 실습 1: 결측치 및 중복값 다루기
- 목표: 작은 데이터셋을 불러와 결측치와 중복값을 확인하고, 초급 수준의 처리를 거친 후 전후 상태를 비교합니다.
- 데이터: 30개 행으로 구성된 사용자 정보 및 점수 데이터

In [2]:
import pandas as pd
import numpy as np

# 1. 파일 불러오기 (첨부 파일명 넣기)
file_name = "data/01_data_missing_and_duplicates.csv"
df_raw = pd.read_csv(file_name)

# 원본 보존을 위한 복사본 생성
df = df_raw.copy()

# 데이터 기본 형태 및 상위 5개 행 확인
print(f"데이터 크기 (행, 열): {df.shape}")
df.head()

데이터 크기 (행, 열): (30, 5)


,user_id,age,city,score,plan
0,1,23.0,Seoul,3.2,Basic
1,2,31.0,Busan,4.1,Premium
2,3,27.0,Incheon,NaN,Basic
3,4,NaN,Seoul,2.8,Basic
4,5,45.0,Daegu,4.7,Premium


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  30 non-null     int64  
 1   age      28 non-null     float64
 2   city     28 non-null     str    
 3   score    27 non-null     float64
 4   plan     27 non-null     str    
dtypes: float64(2), int64(1), str(2)
memory usage: 1.3 KB


In [4]:
df.describe()

,user_id,age,score
count,30.000000,28.000000,27.000000
mean,14.366667,33.821429,3.692593
std,8.083544,7.864847,0.594802
min,1.000000,22.000000,2.700000
25%,7.250000,28.000000,3.250000
50%,14.500000,32.500000,3.600000
75%,20.750000,39.250000,4.100000
max,28.000000,52.000000,4.800000


### Step 1. 결측치 및 중복값 현황 점검
- 데이터셋 내 중복 행 개수를 확인합니다.
- 각 컬럼별 결측치(NaN) 개수를 집계합니다.

In [5]:
# [문제 1] 전체 중복 행 개수 확인하기
# 빈칸 채우기: duplicated() 메서드와 sum()을 활용하세요.
dup_count = df.duplicated().sum()
print(f"중복 행 수: {dup_count}개")
df[df.duplicated(keep=False)]

중복 행 수: 2개


,user_id,age,city,score,plan
6,7,29.0,Seoul,3.5,NaN
17,18,28.0,Seoul,3.6,Standard
28,7,29.0,Seoul,3.5,NaN
29,18,28.0,Seoul,3.6,Standard


In [6]:
# [문제 2] 컬럼별 결측치 개수 집계하기
# 빈칸 채우기: isnull() 또는 isna() 메서드를 활용하세요.
missing_count = df.isnull().sum()
print("\n[컬럼별 결측치]")
print(missing_count)


[컬럼별 결측치]
user_id    0
age        2
city       2
score      3
plan       3
dtype: int64


### Step 2. 중복값 및 결측치 정제
- **중복값**: 완전히 일치하는 중복 행 제거
- **수치형 결측치 (age, score)**: 각 컬럼의 평균값(mean) 또는 중앙값(median)으로 대체
- **범주형 결측치 (city, plan)**: 'Unknown' 또는 최빈값으로 대체

In [9]:
# [문제 3] 중복 행 제거하기
# 빈칸 채우기: 중복 행을 제거하는 판다스 메서드를 입력하세요.
df = df.drop_duplicates()

# [문제 4] 수치형 결측치 채우기 (age: 평균값, score: 중앙값)
age_mean = df['age'].mean()
score_median = df['score'].median()

# fillna()를 활용해 빈칸을 채워보세요.
df['age'] = df['age'].fillna(age_mean)
df['score'] = df['score'].fillna(score_median)

# [문제 5] 범주형 결측치 대체 (city, plan: 'Unknown' 문자열로 대체)
df['city'] = df['city'].fillna('Unknown')
df['plan'] = df['plan'].fillna('Unknown')

# 처리 후 잔여 결측치 확인
print("처리 후 잔여 결측치 합계:", df.isnull().sum().sum())

처리 후 잔여 결측치 합계: 0


### Step 3. 처리 전후 수치 비교
- 정제 전후의 행 개수, 결측치 수, 수치형 변수의 요약 통계량 변화를 비교합니다.

In [15]:
# 1. 크기 및 결측치 비교 표 생성
summary_data = {
    "구분" : ["전체 행 수", "결측치 총합", "중복 행 수"],
    "전처리 전" : [len(df_raw), df_raw.isnull().sum().sum(), df_raw.duplicated().sum()],
    "전처리 후" : [len(df), df.isnull().sum().sum(), df.duplicated().sum()]
}

compare_df = pd.DataFrame(summary_data)
print("=== 전처리 전후 데이터 상태 비교 ===")
display(compare_df)

print("\n=== 수치형 컬럼 요약통계 비교 (전처리 전) ===")
display(df_raw[['age', 'score']].describe())

print("\n=== 수치형 컬럼 요약통계 비교 (전처리 후) ===")
display(df[['age', 'score']].describe())

=== 전처리 전후 데이터 상태 비교 ===


,구분,전처리 전,전처리 후
0,전체 행 수,30,28
1,결측치 총합,10,0
2,중복 행 수,2,0



=== 수치형 컬럼 요약통계 비교 (전처리 전) ===


,age,score
count,28.000000,27.000000
mean,33.821429,3.692593
std,7.864847,0.594802
min,22.000000,2.700000
25%,28.000000,3.250000
50%,32.500000,3.600000
75%,39.250000,4.100000
max,52.000000,4.800000



=== 수치형 컬럼 요약통계 비교 (전처리 후) ===


,age,score
count,28.000000,28.000000
mean,34.230769,3.703571
std,7.718701,0.582130
min,22.000000,2.700000
25%,28.750000,3.275000
50%,34.115385,3.700000
75%,39.250000,4.100000
max,52.000000,4.800000


In [20]:
# 1. 전처리 전/후 describe() 생성 및 접미사 부여
desc_before = df_raw[['age', 'score']].describe().add_suffix('(before)')
desc_after = df[['age', 'score']].describe().add_suffix('(after)')

# 2. 가로로 이어붙이기 (axis=1)
stats_compare = pd.concat([desc_before, desc_after], axis=1)

# 3. 차이값(After - Before) 컬럼 추가
stats_compare['age_diff'] = stats_compare['age(after)'] - stats_compare['age(before)']
stats_compare['score_diff'] = stats_compare['score(after)'] - stats_compare['score(before)']

# 4. 컬럼 순서 재배치 (age 끼리, score 끼리 묶고 diff는 맨 뒤로)
columns_order = [
    'age(before)', 'age(after)',
    'score(before)', 'score(after)',
    'age_diff', 'score_diff'
]
stats_compare = stats_compare[columns_order]

print("=== 수치형 컬럼 요약통계 비교 및 차이 (After - Before) ===")
display(stats_compare.round(3))

=== 수치형 컬럼 요약통계 비교 및 차이 (After - Before) ===


,age(before),age(after),score(before),score(after),age_diff,score_diff
count,28.000,28.000,27.000,28.000,0.000,1.000
mean,33.821,34.231,3.693,3.704,0.409,0.011
std,7.865,7.719,0.595,0.582,-0.146,-0.013
min,22.000,22.000,2.700,2.700,0.000,0.000
25%,28.000,28.750,3.250,3.275,0.750,0.025
50%,32.500,34.115,3.600,3.700,1.615,0.100
75%,39.250,39.250,4.100,4.100,0.000,0.000
max,52.000,52.000,4.800,4.800,0.000,0.000


<details>
<summary><b>🔍 정답 및 해설 (클릭하여 펼치기/접기)</b></summary>

<br>

#### [Cell 4 정답: 현황 집계]
```python
dup_count = df.duplicated().sum()
missing_count = df.isnull().sum()  # 또는 df.isna().sum()
```

해설: 첨부 데이터 기준으로 총 30개 행 중 2개의 완전 중복 행(7번, 18번 user_id)이 탐지되며, 결측치는 age 2개, city 2개, score 3개, plan 3개로 집계됩니다.

#### [Cell 6 정답: 정제 처리]

```python
df = df.drop_duplicates()
df['age'] = df['age'].fillna(age_mean)
df['score'] = df['score'].fillna(score_median)
df['city'] = df['city'].fillna('Unknown')
df['plan'] = df['plan'].fillna('Unknown')
```

해설: 중복 제거(drop_duplicates()) 후 전체 행 수는 30행에서 28행으로 줄어들며, 수치형 변수는 각각 계산된 평균/중앙값으로, 범주형 변수는 'Unknown'으로 채워져 잔여 결측치 합계는 0이 됩니다.